# 06 · Human-in-the-Loop Approval

Pausing an agent before a tool call so a human can approve or reject it.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AfzalManihar/Agentic-AI-Learning/blob/main/06_human_in_the_loop.ipynb)

**Covered in this notebook**

- Types of Agents

> Part of [Agentic AI Learning](https://github.com/AfzalManihar/Agentic-AI-Learning). Add your own API keys in Colab Secrets (key icon) before running.

13-09-2026

# Types of Agents
## Human in the loop Agent

In [ ]:
pip install langchain-groq langgraph langchain-community ddgs -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from google.colab import userdata

my_api_key = userdata.get('groq_api')

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = 'openai/gpt-oss-120b',
    temperature = 0,
    api_key = my_api_key
)

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

def web_search(query: str) -> str:
    """Search the web for latest information"""
    print("🔎 web_search actually ran")
    return search.run(query)

def add(a: float, b: float) -> float:
    """Adds two numbers"""
    print("➕ add actually ran")
    return a + b

def multiply(a: float, b: float) -> float:
    """Multiplies two numbers"""
    print("✖️ multiply actually ran")
    return a * b

In [ ]:
tools = [web_search, add, multiply]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
from langchain_core.messages import SystemMessage

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
def assistant_node(state: State):
    system = SystemMessage(content=
        "You are a helpful assistant."
        "Do not do calculations yourself, use the multiply tool or add tool"
        "Do not answer questions of factual or latest information directly, use web_search tool"
    )

    response = llm_with_tools.invoke([system] + state["messages"])

    return {
        "messages": [response]
    }

def should_continue(state: State) -> Literal["tools", END]:
    if state["messages"][-1].tool_calls:
        return "tools"
    return END

In [ ]:
memory = MemorySaver()

builder = StateGraph(State)
builder.add_node("assistant", assistant_node)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", should_continue)
builder.add_edge("tools", "assistant")

# This is the line that was missing. Halt BEFORE the tools node runs.
hitl_graph = builder.compile(
    checkpointer = memory,
    interrupt_before = ['tools']
)

In [ ]:
thread_id = 'kartik'

query = 'What is price of silver in India?'
config = {"configurable": {"thread_id": thread_id}}
hitl_graph.invoke({"messages": [("user", query)]}, config=config)['messages'][-1].content

''

In [ ]:
hitl_graph.invoke(None, config=config)['messages'][-1].content

🔎 web_search actually ran


'The current market price of silver in India (99.9\u202f% purity) is about **₹245\u202fper gram**【web_search】.'

In [ ]:
# Human in the loop agent stops before using any tools
# But if you call it again, it is taken as a approval from your side
# When calling second time, DO NOT send any message with it

In [ ]:
from langchain_core.messages import ToolMessage

def run_with_approval(query: str, thread_id: str):
    config = {"configurable": {"thread_id": thread_id}}

    # First run. It will stop right before the first tool call.
    hitl_graph.invoke({"messages": [("user", query)]}, config=config)

    # Keep going until the graph has nothing left to do
    while True:
        snapshot = hitl_graph.get_state(config)

        # snapshot.next is empty when the graph has finished
        if not snapshot.next:
            break

        last_message = snapshot.values["messages"][-1]

        print("\n⏸️  PAUSED. The agent wants to call:")
        for call in last_message.tool_calls:
            print(f"   Tool : {call['name']}")
            print(f"   Args : {call['args']}")

        decision = input("Approve? (y/n): ").strip().lower()

        if decision == "y":
            # Passing None means: resume from where you stopped
            hitl_graph.invoke(None, config=config)
        else:
            # Tell the agent it was denied, then let it respond
            denials = [
                ToolMessage(
                    content = "Tool call denied by the human. Answer without it.",
                    tool_call_id = call["id"]
                )
                for call in last_message.tool_calls
            ]
            hitl_graph.update_state(config, {"messages": denials}, as_node="tools")
            hitl_graph.invoke(None, config=config)

    final = hitl_graph.get_state(config).values
    print("\n🤖 Final Answer:")
    print(final["messages"][-1].content)

In [ ]:
run_with_approval("What is 99 * 88?", thread_id="hitl_1")


⏸️  PAUSED. The agent wants to call:
   Tool : multiply
   Args : {'a': 99, 'b': 88}
Approve? (y/n): y
✖️ multiply actually ran

🤖 Final Answer:
99 × 88 = 8712.


In [ ]:
# Test 2: two tool calls. Approve the first, and it continues
# to the next pause automatically without you re-running anything.
run_with_approval(
    "Search the current price of silver per gram in India, then multiply it by 40.",
    thread_id="hitl_2"
)


⏸️  PAUSED. The agent wants to call:
   Tool : web_search
   Args : {'query': 'current price of silver per gram in India'}
Approve? (y/n): y
🔎 web_search actually ran

⏸️  PAUSED. The agent wants to call:
   Tool : multiply
   Args : {'a': 245, 'b': 40}
Approve? (y/n): y
✖️ multiply actually ran

🤖 Final Answer:
The latest information I found shows that the price of silver in India is **₹245 per gram**.  

Multiplying this price by 40 grams:

**₹245 × 40 = ₹9,800**

So, 40 grams of silver would cost approximately **₹9,800**.
